<a href="https://colab.research.google.com/github/Deep5Varshney/Assessment_2/blob/main/Assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install chromadb sentence-transformers rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 17.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 60.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.5 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation:

In [2]:
import chromadb
from rdflib import Graph, Literal, Namespace, RDF, RDFS, URIRef
from sentence_transformers import SentenceTransformer

g = Graph()
EX = Namespace("http://example.org/")

alice = EX.Alice
bob = EX.Bob
acme = EX.AcmeCorp

g.add((alice, RDF.type, EX.Person))
g.add((alice, EX.knows, bob))
g.add((bob, EX.worksAt, acme))
g.add((bob, RDF.type, EX.Person))

sentences = []
for s, p, o in g:
  s_name = s.split("/")[-1]
  p_name = p.split("/")[-1]
  o_name = o.split("/")[-1] if isinstance(o, URIRef) else str(o)

  sentence = f"{s_name} {p_name} {o_name}"
  sentences.append(sentence)

print("Generated RDF Sentences:")
for s in sentences:
  print("-", s)

client = chromadb.Client()
kg_collection = client.get_or_create_collection(name="knowledge_graph_search")

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(sentences).tolist()

ids = [f"triple_{i}" for i in range(len(sentences))]
kg_collection.add(documents=sentences, embeddings=embeddings, ids=ids)

def query_knowledge_graph(query_text):
  query_embedding = model.encode([query_text]).tolist()
  results = kg_collection.query(query_embeddings=query_embedding, n_results=1)
  return results["documents"][0][0]

user_query = "Who does Alice know?"
print(f"\nQuery: {user_query}")
print(f"Retrieved Answer: {query_knowledge_graph(user_query)}")

Generated RDF Sentences:
- Alice 22-rdf-syntax-ns#type Person
- Bob worksAt AcmeCorp
- Alice knows Bob
- Bob 22-rdf-syntax-ns#type Person


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Query: Who does Alice know?
Retrieved Answer: Alice knows Bob
